In [2]:
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
nhl_path = '/Users/yuhaoyan/Documents/MCIT/CIT550/project_data/nhl_dataset'
mp_path = '/Users/yuhaoyan/Documents/MCIT/CIT550/project_data/moneypuck_dataset'

In [5]:
def combine_yearly_csv_files(file_pattern, output_file=None):
    """
    Combine yearly CSV files matching the pattern into a single DataFrame
    """
    
    all_files = glob.glob(file_pattern)
    
    df_list = []
    
    for filename in all_files:
        # Extract year from filename (assuming format like team_2008.csv)
        year = filename.split('_')[-1].split('.')[0]
        
        df = pd.read_csv(filename)
        
        if 'season' not in df.columns:
            df['season'] = year
            
        
        df_list.append(df)
    
    
    combined_df = pd.concat(df_list, ignore_index=True)
    
    
    if output_file:
        combined_df.to_csv(output_file, index=False)
        
    return combined_df


team_df = combine_yearly_csv_files(os.path.join(mp_path,'teams_*.csv'))
skater_df = combine_yearly_csv_files(os.path.join(mp_path,'skaters_*.csv'))
goalie_df = combine_yearly_csv_files(os.path.join(mp_path,'goalies_*.csv'))

In [6]:
def convert_team_abbreviation(mp_abbr):
    """
    Convert MoneyPuck team abbreviation to NHL dataset abbreviation
    """
    
    mapping = {
        'CBJ': 'CBJ',  # Columbus Blue Jackets
        'WSH': 'WSH',  # Washington Capitals
        'WPG': 'WPG',  # Winnipeg Jets
        'N.J': 'NJD',  # New Jersey Devils
        'L.A': 'LAK',  # Los Angeles Kings
        'OTT': 'OTT',  # Ottawa Senators
        'PIT': 'PIT',  # Pittsburgh Penguins
        'FLA': 'FLA',  # Florida Panthers
        'BOS': 'BOS',  # Boston Bruins
        'S.J': 'SJS',  # San Jose Sharks
        'DAL': 'DAL',  # Dallas Stars
        'CGY': 'CGY',  # Calgary Flames
        'ARI': 'ARI',  # Arizona Coyotes
        'PHI': 'PHI',  # Philadelphia Flyers
        'ANA': 'ANA',  # Anaheim Ducks
        'EDM': 'EDM',  # Edmonton Oilers
        'VAN': 'VAN',  # Vancouver Canucks
        'DET': 'DET',  # Detroit Red Wings
        'T.B': 'TBL',  # Tampa Bay Lightning
        'CHI': 'CHI',  # Chicago Blackhawks
        'NYI': 'NYI',  # New York Islanders
        'COL': 'COL',  # Colorado Avalanche
        'NYR': 'NYR',  # New York Rangers
        'MIN': 'MIN',  # Minnesota Wild
        'TOR': 'TOR',  # Toronto Maple Leafs
        'STL': 'STL',  # St. Louis Blues
        'VGK': 'VGK',  # Vegas Golden Knights
        'CAR': 'CAR',  # Carolina Hurricanes
        'BUF': 'BUF',  # Buffalo Sabres
        'NSH': 'NSH',  # Nashville Predators
        'MTL': 'MTL',  # Montreal Canadiens
        'ATL': 'ATL',  # Atlanta Thrashers
    }
    
    return mapping.get(mp_abbr, mp_abbr)

def add_team_id_column(df, team_column='team'):
    """
    Add a team_id column to the DataFrame based on team abbreviations
    and make it the first column
    """
    team_id_mapping = {
        'NJD': 1,
        'PHI': 4,
        'LAK': 26,
        'TBL': 14,
        'BOS': 6,
        'NYR': 3,
        'PIT': 5,
        'DET': 17,
        'SJS': 28,
        'NSH': 18,
        'VAN': 23,
        'CHI': 16,
        'OTT': 9,
        'MTL': 8,
        'MIN': 30,
        'WSH': 15,
        'STL': 19,
        'ANA': 24,
        'PHX': 27,
        'NYI': 2,
        'TOR': 10,
        'FLA': 13,
        'BUF': 7,
        'CGY': 20,
        'COL': 21,
        'DAL': 25,
        'CBJ': 29,
        'WPG': 52,
        'EDM': 22,
        'VGK': 54,
        'CAR': 12,
        'ARI': 53,
        'ATL': 11,
    }
    
    result_df = df.copy()
    
    result_df['team_id'] = result_df[team_column].map(team_id_mapping)
    
    
    columns = list(result_df.columns)
    columns.remove('team_id')
    result_df = result_df[['team_id'] + columns]
    
    return result_df




In [7]:
team_df['team'] = team_df['team'].apply(convert_team_abbreviation)
goalie_df['team'] = goalie_df['team'].apply(convert_team_abbreviation)
skater_df['team'] = skater_df['team'].apply(convert_team_abbreviation)
team_df = add_team_id_column(team_df)

In [8]:
def split_name(skater_df):
    df = skater_df.copy()
    
    # Split the name column into firstName and lastName
    name_parts = df['name'].str.split(' ', n=1, expand=True)
    
    df['firstName'] = name_parts[0]
    df['lastName'] = name_parts[1]
    
    return df

In [9]:
skater_df = split_name(skater_df)
goalie_df = split_name(goalie_df)
skater_df = skater_df.drop('name', axis=1)
goalie_df = goalie_df.drop('name', axis=1)

In [12]:
goalie_df

,playerId,season,team,position,situation,games_played,icetime,xGoals,goals,unblocked_shot_attempts,...,mediumDangerxGoals,highDangerxGoals,lowDangerGoals,mediumDangerGoals,highDangerGoals,blocked_shot_attempts,penalityMinutes,penalties,firstName,lastName
0,8470140,2015,DAL,G,other,43,4436.0,7.54,4.0,120.0,...,2.75,2.86,2.0,2.0,0.0,40.0,0.0,0.0,Kari,Lehtonen
1,8470140,2015,DAL,G,all,43,136751.0,101.59,105.0,2258.0,...,41.09,27.15,38.0,46.0,21.0,639.0,2.0,1.0,Kari,Lehtonen
2,8470140,2015,DAL,G,5on5,43,110645.0,72.72,80.0,1782.0,...,32.09,13.70,32.0,41.0,7.0,500.0,0.0,0.0,Kari,Lehtonen
3,8470140,2015,DAL,G,4on5,43,10805.0,18.52,17.0,310.0,...,4.93,9.94,3.0,1.0,13.0,91.0,0.0,0.0,Kari,Lehtonen
4,8470140,2015,DAL,G,5on4,43,10865.0,2.80,4.0,46.0,...,1.32,0.65,1.0,2.0,1.0,8.0,2.0,1.0,Kari,Lehtonen
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5390,8471679,2018,MTL,G,other,66,5899.0,11.79,15.0,136.0,...,2.99,6.22,8.0,2.0,5.0,32.0,0.0,0.0,Carey,Price
5391,8471679,2018,MTL,G,all,66,232919.0,166.51,163.0,3575.0,...,59.06,51.53,60.0,54.0,49.0,920.0,2.0,1.0,Carey,Price
5392,8471679,2018,MTL,G,5on5,66,189545.0,120.13,118.0,2857.0,...,46.47,27.76,44.0,46.0,28.0,743.0,0.0,0.0,Carey,Price
5393,8471679,2018,MTL,G,4on5,66,18726.0,30.72,26.0,499.0,...,8.29,16.29,6.0,4.0,16.0,135.0,0.0,0.0,Carey,Price


In [4]:
def check_nulls(df):
    """
    Check for null values in a pandas DataFrame and provide details.
    """
    has_nulls = df.isna().any().any()
    
    result = {
        'has_nulls': has_nulls,
        'total_null_count': df.isna().sum().sum(),
        'rows_with_nulls': df.isna().any(axis=1).sum(),
        'percent_rows_with_nulls': (df.isna().any(axis=1).sum() / len(df)) * 100,
        'columns_with_nulls': [],
        'null_counts_by_column': {}
    }
    
    if has_nulls:
        cols_with_nulls = df.columns[df.isna().any()].tolist()
        result['columns_with_nulls'] = cols_with_nulls
        
        for col in cols_with_nulls:
            null_count = df[col].isna().sum()
            null_percent = (null_count / len(df)) * 100
            result['null_counts_by_column'][col] = {
                'count': null_count,
                'percent': null_percent
            }
        
    
    return result

In [10]:
null_info = check_nulls(skater_df)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")

Has nulls: False
Total null values: 0


In [11]:
null_info = check_nulls(team_df)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 3630


{'has_nulls': True,
 'total_null_count': 3630,
 'rows_with_nulls': 1815,
 'percent_rows_with_nulls': 100.0,
 'columns_with_nulls': ['penaltiesFor',
  'penaltiesAgainst',
  'penalitiesFor',
  'penalitiesAgainst'],
 'null_counts_by_column': {'penaltiesFor': {'count': 155,
   'percent': 8.539944903581267},
  'penaltiesAgainst': {'count': 155, 'percent': 8.539944903581267},
  'penalitiesFor': {'count': 1660, 'percent': 91.46005509641874},
  'penalitiesAgainst': {'count': 1660, 'percent': 91.46005509641874}}}

In [12]:
def merge_columns(df, col1, col2, target_col):
    
    df[target_col] = df[col1].fillna(df[col2])
    
    df = df.drop(columns=[col1, col2])
    return df

In [13]:
team_df = merge_columns(team_df, 'penaltiesFor', 'penalitiesFor', 'penalties_for')
team_df = merge_columns(team_df, 'penaltiesAgainst', 'penalitiesAgainst', 'penalties_against')

In [14]:
null_info = check_nulls(team_df)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: False
Total null values: 0


{'has_nulls': False,
 'total_null_count': 0,
 'rows_with_nulls': 0,
 'percent_rows_with_nulls': 0.0,
 'columns_with_nulls': [],
 'null_counts_by_column': {}}

In [15]:
team_df.loc[team_df['team_id'].isnull()]

,team_id,team,season,name,team.1,position,situation,games_played,xGoalsPercentage,corsiPercentage,...,scoreAdjustedUnblockedShotAttemptsAgainst,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,penalties_for,penalties_against


In [16]:
null_info = check_nulls(goalie_df)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")

Has nulls: False
Total null values: 0


In [17]:
team_df = team_df.drop(columns=['name', 'team.1'])

In [18]:
team_df.to_csv(os.path.join(mp_path, 'mp_team.csv'), index=False)
skater_df.to_csv(os.path.join(mp_path, 'mp_skater.csv'), index=False)
goalie_df.to_csv(os.path.join(mp_path, 'mp_goalie.csv'), index=False)       

In [21]:
game_goalie_stats = pd.read_csv(os.path.join(nhl_path, 'game_goalie_stats.csv'))
null_info = check_nulls(game_goalie_stats)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")


Has nulls: True
Total null values: 9181


array([ 33.33333333, 100.        ,  50.        ,  80.        ,
        88.88888889,  66.66666667,  83.33333333,  75.        ,
        57.14285714,  87.5       ,  85.71428571,          nan,
         0.        ,  90.90909091,  77.77777778,  63.63636364,
        90.        ,  93.33333333,  60.        ,  84.61538462,
        71.42857143,  81.81818182,  92.30769231,  91.66666667,
        88.23529412,  72.72727273,  40.        ,  92.85714286,
        62.5       ,  25.        ,  81.25      ,  89.47368421,
        95.23809524,  55.55555556,  86.66666667,  70.        ,
        76.92307692,  78.57142857,  42.85714286,  94.11764706,
        16.66666667,  93.75      ,  94.44444444,  85.        ,
        95.        ,  72.22222222,  76.47058824,  69.23076923,
        86.36363636,  68.75      ,  82.35294118,  73.33333333,
        54.54545455,  94.73684211,  80.95238095,  76.19047619,
        61.53846154,  82.60869565,  86.95652174,  84.21052632,
        90.47619048,  95.65217391,  96.        ,  95.83

In [22]:
game_goals= pd.read_csv(os.path.join(nhl_path, 'game_goals.csv'))
null_info = check_nulls(game_goals)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 6887


/var/folders/_j/zp727js133gc_vnc4vywxcdh0000gn/T/ipykernel_10881/3466794902.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  game_goals= pd.read_csv(os.path.join(nhl_path, 'game_goals.csv'))


array([False, True, nan], dtype=object)

In [31]:
game_officials= pd.read_csv(os.path.join(nhl_path, 'game_officials.csv'))
null_info = check_nulls(game_officials)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: False
Total null values: 0


{'has_nulls': False,
 'total_null_count': 0,
 'rows_with_nulls': 0,
 'percent_rows_with_nulls': 0.0,
 'columns_with_nulls': [],
 'null_counts_by_column': {}}

In [27]:
game_penalties= pd.read_csv(os.path.join(nhl_path, 'game_penalties.csv'))
# game_penalties['year'] = game_penalties['play_id'].str[:4].astype(int)
# game_penalties = game_penalties[(game_penalties['year'] >= 2009) & (game_penalties['year'] <= 2020)]
null_info = check_nulls(game_penalties)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 12181


{'has_nulls': True,
 'total_null_count': 12181,
 'rows_with_nulls': 12181,
 'percent_rows_with_nulls': 9.194595410628018,
 'columns_with_nulls': ['penaltySeverity'],
 'null_counts_by_column': {'penaltySeverity': {'count': 12181,
   'percent': 9.194595410628018}}}

In [28]:
game_plays_players= pd.read_csv(os.path.join(nhl_path, 'game_plays_players.csv'))
null_info = check_nulls(game_plays_players)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: False
Total null values: 0


{'has_nulls': False,
 'total_null_count': 0,
 'rows_with_nulls': 0,
 'percent_rows_with_nulls': 0.0,
 'columns_with_nulls': [],
 'null_counts_by_column': {}}

In [29]:
game_plays= pd.read_csv(os.path.join(nhl_path, 'game_plays.csv'))
null_info = check_nulls(game_plays)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 10464367


{'has_nulls': True,
 'total_null_count': 10464367,
 'rows_with_nulls': 4007923,
 'percent_rows_with_nulls': 79.3564990914813,
 'columns_with_nulls': ['team_id_for',
  'team_id_against',
  'secondaryType',
  'x',
  'y',
  'periodTimeRemaining',
  'st_x',
  'st_y'],
 'null_counts_by_column': {'team_id_for': {'count': 932705,
   'percent': 18.467471427250494},
  'team_id_against': {'count': 932705, 'percent': 18.467471427250494},
  'secondaryType': {'count': 3868513, 'percent': 76.59619418084719},
  'x': {'count': 1134364, 'percent': 22.460300693254112},
  'y': {'count': 1134333, 'percent': 22.459686896164737},
  'periodTimeRemaining': {'count': 193019, 'percent': 3.821758077223198},
  'st_x': {'count': 1134364, 'percent': 22.460300693254112},
  'st_y': {'count': 1134364, 'percent': 22.460300693254112}}}

In [32]:
game_scratches = pd.read_csv(os.path.join(nhl_path, 'game_scratches.csv'))
null_info = check_nulls(game_scratches)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: False
Total null values: 0


{'has_nulls': False,
 'total_null_count': 0,
 'rows_with_nulls': 0,
 'percent_rows_with_nulls': 0.0,
 'columns_with_nulls': [],
 'null_counts_by_column': {}}

In [33]:
game_shifts = pd.read_csv(os.path.join(nhl_path, 'game_shifts.csv'))
null_info = check_nulls(game_shifts)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 2258


{'has_nulls': True,
 'total_null_count': 2258,
 'rows_with_nulls': 2258,
 'percent_rows_with_nulls': 0.019002359305328276,
 'columns_with_nulls': ['shift_end'],
 'null_counts_by_column': {'shift_end': {'count': 2258,
   'percent': 0.019002359305328276}}}

In [39]:
game_skater_stats = pd.read_csv(os.path.join(nhl_path, 'game_skater_stats.csv'))
# game_skater_stats['year'] = game_skater_stats['game_id'].astype(str).str[:4].astype(int)
# game_skater_stats = game_skater_stats[(game_skater_stats['year'] >= 2010) & (game_skater_stats['year'] <= 2020)]
null_info = check_nulls(game_skater_stats)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 1592428


{'has_nulls': True,
 'total_null_count': 1592428,
 'rows_with_nulls': 398107,
 'percent_rows_with_nulls': 42.09075626698244,
 'columns_with_nulls': ['hits', 'takeaways', 'giveaways', 'blocked'],
 'null_counts_by_column': {'hits': {'count': 398107,
   'percent': 42.09075626698244},
  'takeaways': {'count': 398107, 'percent': 42.09075626698244},
  'giveaways': {'count': 398107, 'percent': 42.09075626698244},
  'blocked': {'count': 398107, 'percent': 42.09075626698244}}}

In [40]:
game_teams_stats = pd.read_csv(os.path.join(nhl_path, 'game_teams_stats.csv'))
null_info = check_nulls(game_teams_stats)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 44320


{'has_nulls': True,
 'total_null_count': 44320,
 'rows_with_nulls': 23056,
 'percent_rows_with_nulls': 43.82436799087626,
 'columns_with_nulls': ['head_coach',
  'goals',
  'shots',
  'hits',
  'pim',
  'powerPlayOpportunities',
  'powerPlayGoals',
  'faceOffWinPercentage',
  'giveaways',
  'takeaways',
  'blocked',
  'startRinkSide'],
 'null_counts_by_column': {'head_coach': {'count': 28,
   'percent': 0.0532218209465881},
  'goals': {'count': 8, 'percent': 0.01520623455616803},
  'shots': {'count': 8, 'percent': 0.01520623455616803},
  'hits': {'count': 4928, 'percent': 9.367040486599505},
  'pim': {'count': 8, 'percent': 0.01520623455616803},
  'powerPlayOpportunities': {'count': 8, 'percent': 0.01520623455616803},
  'powerPlayGoals': {'count': 8, 'percent': 0.01520623455616803},
  'faceOffWinPercentage': {'count': 22148, 'percent': 42.09846036875119},
  'giveaways': {'count': 4928, 'percent': 9.367040486599505},
  'takeaways': {'count': 4928, 'percent': 9.367040486599505},
  'block

In [5]:
game = pd.read_csv(os.path.join(nhl_path, 'game.csv'))
null_info = check_nulls(game)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 1196


{'has_nulls': True,
 'total_null_count': 1196,
 'rows_with_nulls': 1196,
 'percent_rows_with_nulls': 4.54666413229424,
 'columns_with_nulls': ['home_rink_side_start'],
 'null_counts_by_column': {'home_rink_side_start': {'count': 1196,
   'percent': 4.54666413229424}}}

In [20]:
game['type'].unique()

array(['R', 'P', 'A'], dtype=object)

In [42]:
player_info = pd.read_csv(os.path.join(nhl_path, 'player_info.csv'))
null_info = check_nulls(player_info)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: True
Total null values: 1162


{'has_nulls': True,
 'total_null_count': 1162,
 'rows_with_nulls': 1135,
 'percent_rows_with_nulls': 28.9171974522293,
 'columns_with_nulls': ['nationality',
  'birthCity',
  'birthStateProvince',
  'height',
  'height_cm',
  'weight',
  'shootsCatches'],
 'null_counts_by_column': {'nationality': {'count': 8,
   'percent': 0.2038216560509554},
  'birthCity': {'count': 5, 'percent': 0.12738853503184713},
  'birthStateProvince': {'count': 1123, 'percent': 28.611464968152866},
  'height': {'count': 3, 'percent': 0.07643312101910828},
  'height_cm': {'count': 3, 'percent': 0.07643312101910828},
  'weight': {'count': 3, 'percent': 0.07643312101910828},
  'shootsCatches': {'count': 17, 'percent': 0.43312101910828027}}}

In [43]:
team_info = pd.read_csv(os.path.join(nhl_path, 'team_info.csv'))
null_info = check_nulls(team_info)
print(f"Has nulls: {null_info['has_nulls']}")
print(f"Total null values: {null_info['total_null_count']}")
null_info

Has nulls: False
Total null values: 0


{'has_nulls': False,
 'total_null_count': 0,
 'rows_with_nulls': 0,
 'percent_rows_with_nulls': 0.0,
 'columns_with_nulls': [],
 'null_counts_by_column': {}}

In [6]:
def remove_composite_duplicates(df, key_columns, keep='first', verbose=True, show_examples=3):
    """
    Remove duplicate rows based on composite key columns.
    
    Parameters:
    df : pandas.DataFrame
        The DataFrame to process
    key_columns : list
        List of column names that form the composite key
    keep : str, default 'first'
        Which duplicates to keep {'first', 'last', False}
    verbose : bool, default True
        Whether to print information about duplicates
    show_examples : int, default 3
        Number of example duplicate groups to display
        
    Returns:
    pandas.DataFrame
        DataFrame with duplicates removed
    dict
        Statistics about the deduplication process
    """
    initial_rows = len(df)
    if verbose:
        print(f"Initial DataFrame has {initial_rows} rows")
    duplicate_mask = df.duplicated(subset=key_columns, keep=False)
    duplicate_rows = df[duplicate_mask]
    if len(duplicate_rows) > 0:
        duplicate_keys = duplicate_rows[key_columns].drop_duplicates()
        num_duplicate_keys = len(duplicate_keys)
        if verbose:
            print(f"Found {len(duplicate_rows)} rows with duplicate keys ({num_duplicate_keys} unique key combinations)")
            if show_examples > 0:
                print("\nExample duplicate groups:")
                examples_shown = 0
                
                for i, key_row in duplicate_keys.head(show_examples).iterrows():
                    filter_condition = True
                    for col in key_columns:
                        filter_condition = filter_condition & (df[col] == key_row[col])
                    example_rows = df[filter_condition]
                    key_values = ", ".join([f"{col}={key_row[col]}" for col in key_columns])
                    print(f"\nDuplicate group {examples_shown+1}: {key_values}")
                    print(example_rows)
                    
                    examples_shown += 1
    else:
        num_duplicate_keys = 0
        if verbose:
            print(f"No duplicates found for key columns: {key_columns}")
    df_unique = df.drop_duplicates(subset=key_columns, keep=keep)
    rows_removed = len(df) - len(df_unique)
    if verbose and rows_removed > 0:
        print(f"\nRemoved {rows_removed} duplicate rows, keeping {keep} occurrence")
        print(f"Final DataFrame has {len(df_unique)} rows")
    stats = {
        'original_rows': initial_rows,
        'final_rows': len(df_unique),
        'rows_removed': rows_removed,
        'duplicate_keys': num_duplicate_keys
    }
    
    return df_unique, stats

In [7]:
unique, stats = remove_composite_duplicates(game, ['game_id'])

Initial DataFrame has 26305 rows
Found 5140 rows with duplicate keys (2570 unique key combinations)

Example duplicate groups:

Duplicate group 1: game_id=2019020001
          game_id    season type         date_time_GMT  away_team_id  \
21160  2019020001  20192020    R  2019-10-02T23:00:00Z             9   
21164  2019020001  20192020    R  2019-10-02T23:00:00Z             9   

       home_team_id  away_goals  home_goals       outcome  \
21160            10           3           5  home win REG   
21164            10           3           5  home win REG   

      home_rink_side_start             venue           venue_link  \
21160                right  Scotiabank Arena  /api/v1/venues/null   
21164                right  Scotiabank Arena  /api/v1/venues/null   

      venue_time_zone_id  venue_time_zone_offset venue_time_zone_tz  
21160    America/Toronto                      -5                EST  
21164    America/Toronto                      -5                EST  

Duplicate grou

In [8]:
unique.to_csv(os.path.join(nhl_path, 'game_unique.csv'), index=False)